#**Task 1**

import libraries



In [72]:
import pandas as pd
import sqlite3
import json

 Connect to database

In [73]:
conn = sqlite3.connect("download_db.db")

Load Data

In [74]:
members = pd.read_sql("SELECT * FROM members", conn)
books = pd.read_sql("SELECT * FROM books", conn)
checkouts = pd.read_sql("SELECT * FROM checkouts", conn)

Load JSON book catalog

In [75]:
with open("download_json.json", "r") as file:
    book_catalog = json.load(file)
book_catalog = pd.DataFrame(book_catalog)

Load HTML Reading Kickoff data

In [76]:
reading_kickoff = pd.read_html("download_html.html")[0]

Answering the questions using queries

In [77]:

# 1) How much is each member borrowing?
q1 = """
SELECT
    members.member_id,
    COUNT(checkouts.book_id) AS total_books_borrowed
FROM members
LEFT JOIN checkouts
ON members.member_id = checkouts.member_id
GROUP BY members.member_id;
"""

answer1 = pd.read_sql(q1, conn)
display(answer1)












,member_id,total_books_borrowed
0,1001,1
1,1002,2
2,1003,9
3,1004,0
4,1005,3
...,...,...
75,1076,7
76,1077,6
77,1078,0
78,1079,10


In [78]:
# 2) Which books match a chosen author pattern?

q2 = """
SELECT *
FROM books
WHERE author LIKE 'A%';
"""

answer2 = pd.read_sql(q2, conn)
display(answer2)


,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez
5,506,The Paper Boat Club,Aya Hafez


In [79]:

# 3) What are the most popular books?

q3 = """
SELECT
    books.title,
    COUNT(checkouts.book_id) AS checkout_count
FROM checkouts
JOIN books
ON checkouts.book_id = books.book_id
GROUP BY books.title
ORDER BY checkout_count DESC
LIMIT 5;
"""

answer3 = pd.read_sql(q3, conn)
display(answer3)

,title,checkout_count
0,The Silver Kite,57
1,Fossils and Fireflies,55
2,Circuits for Beginners,46
3,Kites Over Cairo,38
4,Storms and Sailboats,25


In [80]:
q4 = """
SELECT
    members.member_id,
    COUNT(checkouts.book_id) AS books_borrowed
FROM members
JOIN checkouts
ON members.member_id = checkouts.member_id
GROUP BY members.member_id
ORDER BY books_borrowed DESC
LIMIT 10;
"""

answer4 = pd.read_sql(q4, conn)
display(answer4)

,member_id,books_borrowed
0,1034,25
1,1044,21
2,1008,19
3,1027,18
4,1010,18
5,1065,17
6,1024,17
7,1018,17
8,1047,16
9,1030,16


In [81]:
# 5) Neighborhood activity over time

q5 = """
SELECT
    members.neighborhood,
    checkouts.checkout_date,
    checkouts.book_id
FROM checkouts
JOIN members
ON checkouts.member_id = members.member_id
ORDER BY checkouts.checkout_date ASC;
"""
answer5 = pd.read_sql(q5, conn)
display(answer5)

,neighborhood,checkout_date,book_id
0,Maadi,2024-01-02,509
1,Shubra,2024-01-03,514
2,Zamalek,2024-01-04,521
3,Maadi,2024-01-05,507
4,Heliopolis,2024-01-09,505
...,...,...,...
386,Nasr City,2025-12-24,503
387,Maadi,2025-12-27,520
388,Maadi,2025-12-28,530
389,Nasr City,2025-12-28,531


Combine Sources

In [82]:
combined = checkouts.merge(
    members,
    on="member_id",)

combined = combined.merge(
    books,
    on="book_id", )
combined = pd.concat(
    [combined, reading_kickoff],
    ignore_index=True )

 Check missing values


In [83]:
print("Missing Values:")
print(combined.isnull().sum())







Missing Values:
checkout_id           26
member_id             26
book_id               26
checkout_date         26
return_date           91
first_name            26
last_name             26
grade                 62
neighborhood          26
membership_status     26
join_date             31
title                 26
author                26
Member ID            391
Book ID              391
Checkout Date        391
dtype: int64


Save final file


In [84]:
combined.to_csv("Library_Combined_Dataset.csv", index=False)

#**Task 2**

Load dataset

In [85]:
df = pd.read_csv("Library_Combined_Dataset.csv")
print("Dataset shape:")
print(df.shape)
print("Columns:")
print(df.columns)

Dataset shape:
(417, 16)
Columns:
Index(['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date',
       'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status',
       'join_date', 'title', 'author', 'Member ID', 'Book ID',
       'Checkout Date'],
      dtype='object')


Problem 1: Missing Values

In [86]:
print(df.isnull().sum())


checkout_id           26
member_id             26
book_id               26
checkout_date         26
return_date           91
first_name            26
last_name             26
grade                 62
neighborhood          26
membership_status     26
join_date             31
title                 26
author                26
Member ID            391
Book ID              391
Checkout Date        391
dtype: int64



 Fill missing values

In [87]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())
print(df.isnull().sum())

checkout_id          0
member_id            0
book_id              0
checkout_date        0
return_date          0
first_name           0
last_name            0
grade                0
neighborhood         0
membership_status    0
join_date            0
title                0
author               0
Member ID            0
Book ID              0
Checkout Date        0
dtype: int64


Problem 2: Duplicates

In [88]:
print("Duplicate Rows:")
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.duplicated().sum())



Duplicate Rows:
8
0


 Problem 3: Inconsistent Text Values


In [89]:
text_columns = df.select_dtypes(include="object").columns
for col in text_columns:
    df[col] = df[col].str.strip()
    df[col] = df[col].str.lower()
print(df[text_columns].head())



  checkout_date return_date first_name last_name neighborhood  \
0    2024-10-21  2024-11-07       sara    rashad   heliopolis   
1    2025-08-24  2025-09-01       seif      zaki      zamalek   
2    2024-02-04  2024-02-16       adam    shafik   heliopolis   
3    2025-06-21  2025-06-29       nada      zaki    nasr city   
4    2025-11-11  2025-12-03       rana     osman       shubra   

  membership_status   join_date                    title        author  \
0          inactive  2024-06-25  shadows on the corniche   hani nagati   
1            active  2025-10-21   circuits for beginners  galal mounir   
2            active  2024-01-03    footsteps in the dust  laila shokry   
3            active  2025-10-19   circuits for beginners  galal mounir   
4            active  2024-10-27     winter in alexandria  farida anwar   

  Checkout Date  
0    2025-07-07  
1    2025-07-07  
2    2025-07-07  
3    2025-07-07  
4    2025-07-07  


Problem 4: Invalid Member IDs

In [90]:
if "member_id" in df.columns:
    if "members_id" in globals():
        valid_members = members["member_id"].unique()
        invalid = df[~df["member_id"].isin(valid_members)]
        print("Invalid member IDs:")
        print(invalid)
        df = df[df["member_id"].isin(valid_members)]
print(df.head())

   checkout_id  member_id  book_id checkout_date return_date first_name  \
0       9263.0     1047.0    517.0    2024-10-21  2024-11-07       sara   
1       9340.0     1072.0    513.0    2025-08-24  2025-09-01       seif   
2       9231.0     1053.0    523.0    2024-02-04  2024-02-16       adam   
3       9129.0     1032.0    513.0    2025-06-21  2025-06-29       nada   
4       9370.0     1079.0    511.0    2025-11-11  2025-12-03       rana   

  last_name  grade neighborhood membership_status   join_date  \
0    rashad    8.0   heliopolis          inactive  2024-06-25   
1      zaki    9.0      zamalek            active  2025-10-21   
2    shafik    9.0   heliopolis            active  2024-01-03   
3      zaki    7.0    nasr city            active  2025-10-19   
4     osman    8.0       shubra            active  2024-10-27   

                     title        author  Member ID  Book ID Checkout Date  
0  shadows on the corniche   hani nagati     1052.0    516.5    2025-07-07  
1   

Save cleaned dataset

In [91]:
df.to_csv("task2_cleaned_data.csv", index=False)

#**Task 3**

 Load cleaned dataset

In [92]:
df = pd.read_csv("task2_cleaned_data.csv")

Data Fairness Analysis

 Compare neighborhoods by members and checkouts

In [93]:
fairness_analysis = df.groupby("neighborhood").agg(
    number_of_members=("member_id", "nunique"),
    number_of_checkouts=("book_id", "count")
)
print(fairness_analysis)


              number_of_members  number_of_checkouts
neighborhood                                        
heliopolis                   13                   84
maadi                        19                  106
nasr city                    15                  123
shubra                        5                   34
zamalek                      10                   62


 Find highest and lowest activityl

In [94]:

most_members = fairness_analysis["number_of_members"].idxmax()
least_members = fairness_analysis["number_of_members"].idxmin()

most_checkouts = fairness_analysis["number_of_checkouts"].idxmax()
least_checkouts = fairness_analysis["number_of_checkouts"].idxmin()


print("Neighborhood with most members:", most_members)
print("Neighborhood with least members:", least_members)

print("Neighborhood with most checkouts:", most_checkouts)
print("Neighborhood with least checkouts:", least_checkouts)

Neighborhood with most members: maadi
Neighborhood with least members: shubra
Neighborhood with most checkouts: nasr city
Neighborhood with least checkouts: shubra
